# 03a · 생성원과 질레스피 알고리즘 (Generators and the Gillespie Algorithm)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §4.1 (CTMC 정의: 체류시간+점프 사슬, 생성원 $Q$, 2상태 사슬) · Ross 12e §6.2, §6.3 도입부 ($q_{ij}=v_iP_{ij}$ 표기) · Norris §2.1, §2.6 (Q-행렬, $(\Pi,q)$ 분해), §2.7, §2.8 (폭발 없음, $P(t)=e^{Qt}$ — 정리 3 인용) · Lawler 2e §3.2 (유한 상태 CTMC의 무한소 생성원) |
| 선수 노트북 | 02a, 02b, 01a |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **연속시간 마르코프 사슬은 "상태 $i$에 $\mathrm{Exp}(q_i)$만큼 머문 뒤 $\Pi_{ij}=q_{ij}/q_i$로 점프"하는 과정이고, 이 두 난수를 그대로 뽑는 것이 질레스피 알고리즘이다. 생성원 $Q$는 그 비율들의 표다.**

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.ctmc import (is_generator, embedded_chain, gillespie, gillespie_paths,
                        transition_matrix, stationary_ctmc, time_average_occupation)
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("03a")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (02d) 복합 푸아송 $X(t)=\sum_{i=1}^{N(t)} Y_i$의 평균과 분산은?

<details><summary>정답</summary>

$E[X(t)]=\lambda t\,E[Y]$, $\mathrm{Var}(X(t))=\lambda t\,E[Y^2]$. 분산에 $E[Y]^2$가 아니라 $E[Y^2]$가 들어가는 것이 핵심이다 (점프 수의 무작위성까지 포함).

</details>

**Q2.** (02d) 비율 $\lambda$ 푸아송 과정에서 고정 시각 $t$를 덮는 도착 간격의 길이 기댓값은? 왜 $1/\lambda$가 아닌가?

<details><summary>정답</summary>

$(2-e^{-\lambda t})/\lambda\to 2/\lambda$. 긴 구간일수록 고정된 $t$를 덮을 확률이 크기 때문이다 (길이 편향, 검사 역설).

</details>

**Q3.** (02d) 시각 $t$에서 다음 도착까지 남은 시간(잔여수명)의 분포는? 버스 정류장에서 평균 얼마나 기다리나?

<details><summary>정답</summary>

무기억성 때문에 $\mathrm{Exp}(\lambda)$이고 평균 대기는 $1/\lambda$이다. 간격 평균의 절반 $1/(2\lambda)$가 아니다.

</details>

**Q4 (연결 고리).** (02c) 비율 $\lambda$ 푸아송 과정의 각 점을 독립적으로 확률 $p$로 남기면 남은 점들과 버린 점들은 각각 무엇이고 서로 어떤 관계인가?

<details><summary>정답</summary>

남은 점 $\sim$ Poisson 과정(비율 $p\lambda$), 버린 점 $\sim$ Poisson 과정(비율 $(1-p)\lambda$), 두 과정은 독립. — 오늘의 "경쟁하는 시계" 구성은 이 결과를 거꾸로 읽는 것이다: 비율 $q_{i1},q_{i2},\dots$의 독립 알람들을 합치면 비율 $q_i$의 알람 하나이고, 어느 알람인지는 비율에 비례해 독립적으로 정해진다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **연속시간 마르코프 사슬(continuous-time Markov chain, CTMC).** 유한 상태공간 $S$ 위의 과정 $(X_t)_{t\ge0}$로, 모든 $s,t\ge0$, 상태 $i,j$, 시각 $s$까지의 과거 $\mathcal F_s$에 대해
  $$P(X_{t+s}=j\mid X_s=i,\ \mathcal F_s)=P_{ij}(t)$$
  (마르코프 성질 + 시간 동질성). $P(t)=(P_{ij}(t))$를 **전이확률 함수(transition function)** 라 한다. (Durrett 3e §4.1)
- **생성원(generator, Q-행렬) $Q$.** $i\ne j$이면 $q_{ij}\ge0$ ($i\to j$ 전이 **비율**), $q_i:=\sum_{j\ne i}q_{ij}$ (총 이탈률), $Q_{ii}=-q_i$. 따라서 **행합은 0**이다. 무한소 정의: $P(h)=I+Qh+o(h)$, 즉 $P_{ij}(h)=q_{ij}h+o(h)$ ($i\ne j$). (Norris §2.1; Lawler 2e §3.2는 $A$로 표기)
- **체류시간(holding time)과 점프 사슬(jump chain).** 상태 $i$에 머무는 시간 $\sim\mathrm{Exp}(q_i)$ (평균 $1/q_i$), 다음 상태는 $\Pi_{ij}=q_{ij}/q_i$ ($j\ne i$), $\Pi_{ii}=0$. $\Pi$는 확률행렬이며 `spkit.ctmc.embedded_chain(Q)`가 $(\Pi,q)$를 돌려준다. Ross 12e §6.2의 표기로는 $v_i=q_i$, $P_{ij}=\Pi_{ij}$, $q_{ij}=v_iP_{ij}$.
- **경쟁하는 시계(competing clocks) 구성.** 상태 $i$에서 각 $j\ne i$마다 독립 알람 $E_j\sim\mathrm{Exp}(q_{ij})$를 켠다. 가장 먼저 울린 $j$로 이동하고 그 시각까지 머문다. $q_{ij}=0$이면 알람이 없다.
- **흡수 상태(absorbing state).** $q_i=0$인 상태. 체류시간이 $\infty$이다 — 코드에서 `rng.exponential(1/0)`을 부르면 안 되므로 분기가 필요하다.
- **질레스피 알고리즘(stochastic simulation algorithm, SSA; Gillespie 1977).** 현재 상태 $i$에서 $\tau\sim\mathrm{Exp}(q_i)$, $j\sim\Pi_{i\cdot}$를 뽑아 $t\leftarrow t+\tau$, $X\leftarrow j$를 반복한다. 근사가 아니라 **정확한** 표본경로 생성이다.
- **2상태 사슬.** 상태 $\{0,1\}$, $0\to1$ 비율 $\alpha$, $1\to0$ 비율 $\beta$:
  $$Q_2=\begin{pmatrix}-\alpha&\alpha\\ \beta&-\beta\end{pmatrix},\qquad\text{이 노트북의 기본값 }\alpha=1,\ \beta=2.$$
- **3상태 '기계' 사슬 $Q_3$** (03b, 03f에서 재사용). 상태 0=정상, 1=성능저하, 2=고장.
  $$Q_3=\begin{pmatrix}-1.5&1.0&0.5\\ 1.0&-2.0&1.0\\ 2.0&0&-2.0\end{pmatrix}$$
  (정상→저하 1, 정상→고장 0.5, 저하→정상 1, 저하→고장 1, 고장→정상(수리) 2).

### 2.2 정리 1 — 경쟁하는 시계 = 체류시간 + 점프 사슬 (Durrett 3e §4.1, Norris §2.6)

> $E_1,\dots,E_m$이 독립이고 $E_j\sim\mathrm{Exp}(\lambda_j)$, $\Lambda=\sum_j\lambda_j$이면
> $$M=\min_j E_j\sim\mathrm{Exp}(\Lambda),\qquad P(\arg\min=j)=\frac{\lambda_j}{\Lambda},\qquad M\ \text{과}\ \arg\min\ \text{은 독립}.$$
> 따라서 상태 $i$에서 경쟁하는 시계 구성은 체류시간 $\mathrm{Exp}(q_i)$와 그와 **독립인** 점프 $\Pi_{ij}=q_{ij}/q_i$를 준다.

**가정이 왜 필요한가**
- **독립성.** 알람들이 종속이면 최소의 분포가 지수가 아니고 $\arg\min$과 $M$의 독립도 깨진다 (예: $E_2=2E_1$이면 항상 1이 이기고 $M=E_1$).
- **지수분포.** 무기억성이 없으면 '언제 울리는가'와 '누가 울리는가'가 얽힌다 (E3에서 균등 체류시간으로 깨뜨려 본다).
- **$\Lambda<\infty$ (유한 상태).** 무한히 많은 알람이면 $M=0$이 될 수 있다 (폭발, Norris §2.7).

<details><summary>증명 (완전 증명)</summary>

(i) 생존함수: 독립성으로
$$P(M>t)=\prod_j P(E_j>t)=\prod_j e^{-\lambda_j t}=e^{-\Lambda t},$$
즉 $M\sim\mathrm{Exp}(\Lambda)$.

(ii) 결합 사건: $\{\arg\min=j,\ M>t\}=\{E_j>t,\ E_k>E_j\ \forall k\ne j\}$. $E_j$의 밀도 $\lambda_je^{-\lambda_js}$로 조건부를 취하면
$$P(\arg\min=j,\ M>t)=\int_t^\infty \lambda_j e^{-\lambda_j s}\prod_{k\ne j}e^{-\lambda_k s}\,ds=\int_t^\infty \lambda_j e^{-\Lambda s}\,ds=\frac{\lambda_j}{\Lambda}e^{-\Lambda t}.$$

(iii) $t=0$을 넣으면 $P(\arg\min=j)=\lambda_j/\Lambda$. 그러면 (ii)는 $P(\arg\min=j,\ M>t)=P(\arg\min=j)\,P(M>t)$ — 모든 $j,t$에서 곱 형태이므로 $\arg\min$과 $M$은 독립이다.

(iv) 상태 $i$에서 $\lambda_j=q_{ij}$, $\Lambda=q_i$를 대입하면 체류시간 $\mathrm{Exp}(q_i)$, 점프 확률 $q_{ij}/q_i$, 둘의 독립을 얻는다. (02a에서 본 "지수 최소" 결과를 CTMC 언어로 다시 쓴 것이다.) $\square$

</details>

### 2.3 정리 2 — 2상태 사슬의 전이확률 (Durrett 3e §4.1, Ross 12e §6.2)

> $0\to1$ 비율 $\alpha$, $1\to0$ 비율 $\beta$인 2상태 사슬에서
> $$P_{01}(t)=\frac{\alpha}{\alpha+\beta}\left(1-e^{-(\alpha+\beta)t}\right),\quad P_{00}(t)=\frac{\beta}{\alpha+\beta}+\frac{\alpha}{\alpha+\beta}e^{-(\alpha+\beta)t},\quad P_{11}(t)=\frac{\alpha}{\alpha+\beta}+\frac{\beta}{\alpha+\beta}e^{-(\alpha+\beta)t}.$$
> $t\to\infty$에서 $\pi=(\beta,\alpha)/(\alpha+\beta)$.

**가정이 왜 필요한가**
- **시간 동질성.** $P_{01}(t+h)$를 $P_{01}(t)$와 길이 $h$ 구간의 확률로 분해할 때 구간의 위치가 무관해야 한다.
- **무한소 확률 $P_{01}(h)=\alpha h+o(h)$, $P_{10}(h)=\beta h+o(h)$.** 체류시간이 지수이므로 길이 $h$ 안에 두 번 이상 점프할 확률이 $O(h^2)$ — 이것이 $o(h)$ 항을 정당화한다.

<details><summary>증명 (완전 증명)</summary>

$p(t)=P_{01}(t)$라 하자. 시각 $t$의 상태로 조건을 걸고 마르코프 성질과 시간 동질성을 쓰면
$$p(t+h)=p(t)\,P_{11}(h)+(1-p(t))\,P_{01}(h)=p(t)(1-\beta h)+(1-p(t))\alpha h+o(h).$$
정리하고 $h\to0$:
$$p'(t)=\alpha-(\alpha+\beta)p(t),\qquad p(0)=0.$$
선형 1계 ODE이므로 적분인자 $e^{(\alpha+\beta)t}$를 곱하면
$$\frac{d}{dt}\left[e^{(\alpha+\beta)t}p(t)\right]=\alpha e^{(\alpha+\beta)t}\ \Rightarrow\ e^{(\alpha+\beta)t}p(t)=\frac{\alpha}{\alpha+\beta}\left(e^{(\alpha+\beta)t}-1\right),$$
즉 $p(t)=\frac{\alpha}{\alpha+\beta}(1-e^{-(\alpha+\beta)t})$. $P_{00}=1-P_{01}$이고, $\alpha\leftrightarrow\beta$ 대칭으로 $P_{11}$을 얻는다. 극한은 $\pi_1=\alpha/(\alpha+\beta)$, $\pi_0=\beta/(\alpha+\beta)$. **수렴 속도는 $\alpha+\beta$(두 비율의 합)** 임에 주의. $\square$

</details>

### 2.4 정리 3 — 구성된 과정은 CTMC이고 무한소 생성원은 $Q$ (Norris §2.8; 폭발 없음은 §2.7)

> 체류시간 $\mathrm{Exp}(q_i)$와 점프 사슬 $\Pi$로 만든 과정은 마르코프 성질을 만족하며 $P(h)=I+Qh+o(h)$를 만족한다. 유한 상태공간에서는 유한 시간에 무한히 많은 점프(폭발, explosion)가 일어나지 않는다.

**가정이 왜 필요한가**
- **지수 체류시간의 무기억성.** 임의 시각 $t$에서 남은 체류시간이 과거와 무관해야 마르코프 성질이 성립한다.
- **유한 상태공간.** $\max_i q_i<\infty$이면 점프 사이 시간이 $\mathrm{Exp}(\max_iq_i)$에 의해 확률적으로 아래에서 지배되어 폭발이 없다.

**증명은 인용** (Norris §2.8 참고: 유한 상태공간에서 $(\Pi,q)$ 구성 $\Leftrightarrow$ $P(t)=e^{Qt}$; 폭발 없음은 Norris §2.7). 핵심 아이디어만: 시각 $t$에 상태 $i$에 있으면 무기억성으로 남은 체류시간이 다시 $\mathrm{Exp}(q_i)$이고 다음 점프는 $\Pi_{i\cdot}$ — 과거는 필요 없다. $P(t)=e^{Qt}$는 03b에서 콜모고로프 방정식으로 유도한다.

### 2.5 직관

생성원 $Q$의 $i$번째 행은 **'상태 $i$에 놓인 알람 시계들의 속도표'** 다. 비대각 원소 $q_{ij}$는 $j$로 가는 알람의 비율, 대각 $-q_i$는 그 합에 음의 부호를 붙여 행합을 0으로 만든 것. 평균 체류시간은 $1/q_i$.

DTMC(01a)와의 차이 세 가지:
1. 시간이 연속이라 '한 스텝'이 없고 대신 **비율**이 있다.
2. **자기 전이가 무의미**하다 — 머무는 것은 이미 체류시간이 표현한다 ($\Pi_{ii}=0$).
3. **주기성이 없다** (03b).

질레스피 알고리즘은 '다음 사건이 **언제**($\mathrm{Exp}$), **무엇**인지($\Pi$)'를 정확히 뽑기 때문에 시간 격자 근사와 달리 이산화 오차가 0이다. 2상태 사슬에서 $P_{01}(t)$가 $\pi_1$로 수렴하는 속도 $\alpha+\beta$는 '양방향 알람이 모두 평형을 재촉한다'로 읽는다.

### 2.6 함정 (traps)

- $Q$의 대각원소 $-q_i$는 확률이 아니다. $Q$를 전이확률처럼 곱하면 안 되고 $P(h)\approx I+Qh$만 성립한다 (정확한 값은 03b의 $e^{Qh}$).
- numpy의 `rng.exponential(scale)`은 **평균(scale)** 을 받는다. 체류시간은 `rng.exponential(1/q_i)`이지 `rng.exponential(q_i)`가 아니다.
- 점프 사슬 $\Pi$의 정상분포 $\nu$는 CTMC의 정상분포 $\pi$와 다르다 ($\pi_i\propto\nu_i/q_i$, 03c). $\Pi$로 장기 점유율을 계산하면 틀린다.
- 시간 격자 $\Delta t$ 위에서 $I+Q\Delta t$로 굴리는 것은 근사이며 $q_i\Delta t<1$이어야 확률행렬이다. 질레스피는 정확하다.
- 2상태 사슬의 수렴률은 $\alpha+\beta$이지 $\max(\alpha,\beta)$나 $\min$이 아니다.
- `gillespie`가 돌려주는 마지막 점프 시각이 $T$를 넘을 수 있다 (마지막 상태를 $T$까지 유지). $[0,T]$의 점프 수는 `(times <= T).sum() - 1`로 세어야 한다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — (a) $Q_3$의 0번째 행을 알람 시계 이야기로 설명하고, (b) 왜 대각원소가 음수여야 하는지, (c) 질레스피 한 스텝에서 난수가 몇 개 필요한지 써 보세요.)

## 3. Predict — 코드를 돌리기 전에 예측

아래 다섯 값을 손으로(또는 암산으로) 먼저 적어 보세요. 정답은 5절에서 코드가 계산합니다. 틀려도 괜찮습니다 — 예측과 결과의 차이가 학습 신호입니다.

In [ ]:
predictions = {
    # Q3의 상태 0(정상)에 한 번 머무는 평균 시간은?
    "mean_hold_0": None,
    # 상태 0을 떠날 때 성능저하(1)로 갈 확률은?
    "p_jump_0_to_1": None,
    # 2상태 사슬(alpha=1, beta=2)이 0에서 출발해 t=0.5에 1에 있을 확률은?
    "p01_at_half": None,
    # 같은 2상태 사슬이 0에서 출발할 때 [0, 10] 동안 일어나는 점프 수의 기댓값은?
    "mean_jumps_T10": None,
    # Q3 기계가 아주 오랜 시간 동안 고장(2) 상태에 있는 시간 비율은?
    "pi2_three_state": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 생성원 정의와 검사

`Q3`를 2.1의 정의대로 만들고, 생성원 조건(비대각 $\ge0$, 행합 0)과 `embedded_chain`이 주는 $(\Pi,q)$를 확인한다. 예상: $\Pi=\begin{pmatrix}0&2/3&1/3\\1/2&0&1/2\\1&0&0\end{pmatrix}$, $q=(1.5,2,2)$.

In [ ]:
Q3 = np.array([[-1.5, 1.0, 0.5],
               [ 1.0, -2.0, 1.0],
               [ 2.0, 0.0, -2.0]])
alpha, beta = 1.0, 2.0
Q2 = np.array([[-alpha, alpha],
               [ beta, -beta]])
print("is_generator(Q3) =", is_generator(Q3), " is_generator(Q2) =", is_generator(Q2))
print("row sums Q3:", Q3.sum(axis=1))

Pi3, q3 = embedded_chain(Q3)
np.testing.assert_allclose(Pi3.sum(axis=1), 1.0)   # 점프 행렬은 확률행렬
Pi2, q2 = embedded_chain(Q2)
print("holding rates q (Q3):", q3, "  mean holding times 1/q:", np.round(1 / q3, 4))
print("jump matrix Pi (Q3):\n", np.round(Pi3, 4))
print("Q2: q =", q2, " Pi =", Pi2.tolist())

$\Pi_3$의 대각이 모두 0이고, 상태 2(고장)는 수리되면 반드시 정상(0)으로 돌아가므로 $\Pi_{20}=1$이다.

### 4.2 한 스텝을 두 방식으로 — 정리 1 확인

상태 0에서 (a) **경쟁하는 시계**: $E_1\sim\mathrm{Exp}(q_{01}=1)$, $E_2\sim\mathrm{Exp}(q_{02}=0.5)$를 켜고 먼저 울린 쪽으로; (b) **체류 + 점프**: $\mathrm{Exp}(q_0=1.5)$ 뒤 $(2/3,1/3)$로. 두 방식이 같은 평균 체류시간, 같은 $P(\text{next}=1)$을 주는지, 그리고 (a)에서 체류시간과 목적지가 **독립**인지(어디로 가든 평균 체류가 $\approx1/1.5$) 본다.

In [ ]:
n = N_PATHS
# (a) 경쟁하는 시계: 열 0 = 상태 1로 가는 알람, 열 1 = 상태 2로 가는 알람
clocks = np.column_stack([rng.exponential(1 / Q3[0, 1], n),   # Exp(1.0)  -> 상태 1
                          rng.exponential(1 / Q3[0, 2], n)])  # Exp(0.5)  -> 상태 2
hold_a = clocks.min(axis=1)
next_a = clocks.argmin(axis=1) + 1
# (b) 체류시간 + 점프 사슬
hold_b = rng.exponential(1 / q3[0], n)
next_b = rng.choice(3, size=n, p=Pi3[0])

print(f"(a) competing clocks : mean hold = {mc_estimate(hold_a)},  P(next=1) = {mc_proportion(next_a == 1)}")
print(f"(b) holding + jump   : mean hold = {mc_estimate(hold_b)},  P(next=1) = {mc_proportion(next_b == 1)}")
print(f"(a) independence check: E[hold | next=1] = {hold_a[next_a == 1].mean():.4f}, "
      f"E[hold | next=2] = {hold_a[next_a == 2].mean():.4f}  (both should be ~ {1 / q3[0]:.4f})")

In [ ]:
x = np.linspace(0, 5, 300)
ax = plot_hist_vs_pdf(hold_a, x, q3[0] * np.exp(-q3[0] * x), bins=60, label_pdf="Exp(1.5) pdf")
ax.set_title("Holding time in state 0: competing clocks (histogram) vs Exp(1.5) pdf")
ax.set_xlabel("holding time"); ax.set_ylabel("density"); ax.set_xlim(0, 5)
plt.show()

경쟁하는 두 알람($\mathrm{Exp}(1)$, $\mathrm{Exp}(0.5)$)의 최소가 정확히 $\mathrm{Exp}(1.5)$를 따르고, 어느 알람이 이겼든 체류시간의 평균이 같다 — 정리 1의 세 주장 모두.

### 4.3 질레스피 알고리즘 직접 구현

`spkit.ctmc.gillespie`와 같은 반환 형식 `(times, states)`를 주는 15줄 구현. 한 스텝에 난수 두 개: 체류시간(`rng.exponential(1/q)`)과 점프(`rng.choice`). 흡수 상태($q_i=0$)에서는 분기해서 멈춘다.

In [ ]:
# helper (spkit에 없어서 여기서 정의) — 교육용 질레스피(SSA)
def gillespie_simple(Q, x0, T, rng):
    """교육용 질레스피(SSA). spkit.ctmc.gillespie와 같은 반환 형식 (times, states)."""
    Q = np.asarray(Q, float); n = len(Q)
    t, x = 0.0, int(x0); times, states = [0.0], [x]
    while True:
        q = -Q[x, x]
        if q <= 0:            # 흡수 상태: 알람이 없다 -> 영원히 머문다
            break
        t += rng.exponential(1.0 / q)          # 체류시간 ~ Exp(q): scale = 평균 = 1/q
        if t > T:
            break
        probs = Q[x].copy(); probs[x] = 0.0; probs /= q   # 점프 사슬 Pi의 x번째 행
        x = int(rng.choice(n, p=probs))
        times.append(t); states.append(x)
    return np.array(times), np.array(states)

먼저 직접 구현으로 경로 하나를 그리고, 그 아래에 `gillespie_paths`(spkit)로 만든 10개 경로를 겹쳐 그린다. 두 구현의 $[0,10]$ 평균 점프 수도 나란히 출력해 같은 통계량을 주는지 본다.

In [ ]:
T_path = 10.0
times1, states1 = gillespie_simple(Q2, 0, T_path, rng)
t_grid10, X10 = gillespie_paths(Q2, 0, T_path, rng, n_paths=10, n_grid=1001)

fig, axes = plt.subplots(2, 1, figsize=(7, 5.5), sharex=True)
axes[0].step(np.append(times1, T_path), np.append(states1, states1[-1]), where="post", color="C0")
axes[0].plot(times1[1:], states1[1:], "o", ms=4, color="C3", label="jumps")
axes[0].set_title(f"One path from gillespie_simple (Q2, alpha=1, beta=2): {len(times1) - 1} jumps in [0, 10]")
axes[0].set_yticks([0, 1]); axes[0].legend(loc="center right")
plot_paths(t_grid10, X10, ax=axes[1], n_show=10, alpha=0.5, step=True)
axes[1].set_title("Two-state CTMC sample paths, alpha=1, beta=2 (spkit gillespie_paths, 10 paths)")
axes[1].set_yticks([0, 1]); axes[1].set_ylabel("state")
plt.tight_layout(); plt.show()

n_chk = scale(2_000)
jumps_simple = np.array([len(gillespie_simple(Q2, 0, T_path, rng)[0]) - 1 for _ in range(n_chk)])
jumps_spkit = np.array([(gillespie(Q2, 0, T_path, rng)[0] <= T_path).sum() - 1 for _ in range(n_chk)])
print(f"mean jumps in [0, 10]: gillespie_simple = {mc_estimate(jumps_simple)},  spkit = {mc_estimate(jumps_spkit)}")

경로는 계단 함수(오른쪽 연속)다. 상태 1에서는 $\beta=2$로 빨리 떠나므로 1에 머무는 구간이 0에 머무는 구간보다 짧다 — 장기적으로 $\pi=(2/3,1/3)$의 비율이 되는 이유다.

### 4.4 $P_{01}(t)$ 곡선 검증 — 정리 2

0에서 출발한 많은 경로를 격자 위에서 관측해, 각 시각에 상태 1에 있는 **경로의 비율**을 정리 2의 곡선과 겹쳐 그린다. $t=0.5$에서의 비율은 5절에서 `p01_at_half`와 비교한다. 또한 `transition_matrix(Q2, 0.5)` $=e^{0.5Q_2}$의 $(0,1)$ 원소가 닫힌 형식과 일치하는지 확인한다 (03b 예고).

In [ ]:
T4, n4 = 2.0, scale(8_000)
t_grid, X = gillespie_paths(Q2, 0, T4, rng, n_paths=n4, n_grid=201)
frac1 = (X == 1).mean(axis=0)

lam = alpha + beta
pi1 = alpha / lam
p01_curve = pi1 * (1 - np.exp(-lam * t_grid))
idx = np.argmin(np.abs(t_grid - 0.5))
p01_half_mc = mc_proportion(X[:, idx] == 1)
p01_half_exact = pi1 * (1 - np.exp(-lam * 0.5))
p01_half_expm = transition_matrix(Q2, 0.5)[0, 1]
np.testing.assert_allclose(p01_half_expm, p01_half_exact, rtol=1e-10)
print(f"t = {t_grid[idx]:.3f}: Gillespie fraction = {p01_half_mc},  closed form = {p01_half_exact:.6f},  expm = {p01_half_expm:.6f}")

fig, ax = plt.subplots()
ax.plot(t_grid[::5], frac1[::5], "o", ms=4, color="C0", label=f"Gillespie fraction (n={n4})")
ax.plot(t_grid, p01_curve, color="C1", lw=2, label="closed form (Theorem 2)")
ax.axhline(pi1, ls="--", color="gray", label="pi_1 = 1/3")
ax.set_title("P(X_t = 1 | X_0 = 0) for the two-state chain, alpha=1, beta=2")
ax.set_xlabel("t"); ax.set_ylabel("probability"); ax.legend(loc="lower right")
plt.show()

점(시뮬레이션)이 곡선(닫힌 형식)을 따라가고, $t\approx 1.5$ ($=4.5/(\alpha+\beta)$, $e^{-4.5}\approx0.01$)면 이미 $\pi_1=1/3$에 붙는다. 수렴 속도가 $\alpha+\beta=3$이라는 뜻이다.

### 4.5 $[0,10]$ 동안의 점프 수

점프 강도(시각 $s$에 점프가 일어나는 비율)는 $\alpha P_{00}(s)+\beta P_{01}(s)$이므로 기댓값은 이를 $[0,T]$에서 적분한 것이다 (5절에서 계산). `gillespie`는 마지막 점프 시각이 $T$를 넘을 수 있으므로 `(times <= T).sum() - 1`로 센다.

In [ ]:
T5, n5 = 10.0, scale(2_000)
n_jumps = np.empty(n5)
for i in range(n5):
    times, states = gillespie(Q2, 0, T5, rng)
    n_jumps[i] = (times <= T5).sum() - 1
jumps_mc = mc_estimate(n_jumps)
print("mean number of jumps in [0, 10]:", jumps_mc)

fig, ax = plt.subplots()
bins = np.arange(n_jumps.min() - 0.5, n_jumps.max() + 1.5)
ax.hist(n_jumps, bins=bins, density=True, alpha=0.6, label="simulation")
ax.axvline(jumps_mc.mean, color="C1", lw=2, label=f"MC mean = {jumps_mc.mean:.2f}")
ax.set_title("Number of jumps in [0, 10] for the two-state chain")
ax.set_xlabel("number of jumps"); ax.set_ylabel("density"); ax.legend()
plt.show()

점프 수는 푸아송이 **아니다** — 체류율이 상태에 따라 다르기 때문(0에서 1, 1에서 2). 그래도 평균은 점프 강도의 적분으로 정확히 계산된다.

### 4.6 3상태 기계의 장기 점유율 (03c 예고)

정상분포 $\pi$는 $\pi Q_3=0$, $\sum\pi=1$의 해다 (`stationary_ctmc`). 각 경로의 **시작 상태를 $\pi$에서 뽑으면** 과정이 처음부터 정상 상태이므로 시간평균 점유율의 기댓값이 정확히 $\pi$가 된다(초기 편향 0). 경로별 점유율 벡터를 모아 평균과 SE를 구한다.

In [ ]:
pi3 = stationary_ctmc(Q3)
np.testing.assert_allclose(pi3 @ Q3, 0.0, atol=1e-12)
T6, n6 = (20.0, scale(400)) if fast() else (50.0, 400)
occ = np.empty((n6, 3))
x0s = rng.choice(3, size=n6, p=pi3)        # 정상분포에서 시작
for i in range(n6):
    times, states = gillespie(Q3, x0s[i], T6, rng)
    occ[i] = time_average_occupation(times, states, T6, 3)
occ_mean, occ_se = occ.mean(axis=0), occ.std(axis=0, ddof=1) / np.sqrt(n6)
frac_state2 = occ[:, 2]
print("stationary pi =", np.round(pi3, 4))
print("time-average occupation (mean over paths) =", np.round(occ_mean, 4), " SE =", np.round(occ_se, 4))

fig, ax = plt.subplots()
labels = ["0 (up)", "1 (degraded)", "2 (down)"]
ax.bar(np.arange(3) - 0.18, occ_mean, 0.36, yerr=1.96 * occ_se, capsize=4, label=f"time average, {n6} paths x T={T6:g}")
ax.bar(np.arange(3) + 0.18, pi3, 0.36, alpha=0.6, label="stationary pi")
ax.set_xticks(range(3), labels); ax.set_ylabel("fraction of time")
ax.set_title("Time-average occupation vs stationary pi for the 3-state machine chain"); ax.legend()
plt.show()

시간평균 점유율은 정상분포 $\pi=(1/2,1/4,1/4)$와 일치한다. 점프 사슬 $\Pi_3$의 정상분포로 계산하면 다른 값이 나온다(함정 3) — 03c에서 $\pi_i\propto\nu_i/q_i$로 정리한다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식 값은 모두 **공식에서 코드로** 계산한다 (숫자를 손으로 옮겨 적지 않는다).

In [ ]:
pi0 = beta / lam
exact = {
    "mean_hold_0": 1 / q3[0],                                # 1/q_0
    "p_jump_0_to_1": Q3[0, 1] / q3[0],                       # Pi_01 = q_01/q_0
    "p01_at_half": pi1 * (1 - np.exp(-lam * 0.5)),           # 정리 2
    # 점프 강도 alpha*P00(s) + beta*P01(s) 를 [0,T]에서 적분
    "mean_jumps_T10": (alpha * pi0 + beta * pi1) * T5 + (alpha - beta) * pi1 * (1 - np.exp(-lam * T5)) / lam,
    "pi2_three_state": pi3[2],                               # pi Q3 = 0 의 해
}
# 점프 수 기댓값의 독립 확인: 점프 강도를 수치 적분 (P00, P01 은 정리 2; expm 과 일치하는지도 확인)
s = np.linspace(0, T5, 20_001)
P00_s, P01_s = pi0 + pi1 * np.exp(-lam * s), pi1 * (1 - np.exp(-lam * s))
np.testing.assert_allclose(transition_matrix(Q2, 1.0)[0], [P00_s[2000], P01_s[2000]], rtol=1e-10)
print("mean jumps: formula =", round(exact["mean_jumps_T10"], 6),
      " trapezoid of alpha*P00(s)+beta*P01(s) =", round(np.trapezoid(alpha * P00_s + beta * P01_s, s), 6))
for k, v in exact.items():
    print(f"{k:16s} exact = {v:.6f}")

In [ ]:
estimates = {
    "mean_hold_0": mc_estimate(hold_a),
    "p_jump_0_to_1": mc_proportion(next_a == 1),
    "p01_at_half": p01_half_mc,
    "mean_jumps_T10": jumps_mc,
    "pi2_three_state": mc_estimate(frac_state2),
}
rows = [{"name": k, "predicted": predictions[k], "estimate": estimates[k], "exact": exact[k]} for k in exact]
Markdown(compare_table(rows))

각 행에서 불일치(`investigate`)가 나오면 뜻하는 바:
- **mean_hold_0** — 경쟁하는 시계의 최소가 $\mathrm{Exp}(q_0)$가 아니다: `rng.exponential`에 비율을 넣었거나(평균이어야 함) 알람이 독립이 아니다.
- **p_jump_0_to_1** — $\arg\min$의 분포가 $q_{0j}/q_0$가 아니다: 정리 1(ii)가 깨졌거나 알람 비율을 잘못 넣었다.
- **p01_at_half** — 시뮬레이션이 정리 2의 ODE 해를 재현하지 못한다: 격자 표본화(`searchsorted`)나 체류시간 분포가 잘못됐다.
- **mean_jumps_T10** — 점프 수 세기 오류(마지막 점프가 $T$를 넘는 경우 포함) 또는 점프 강도 적분 공식 오류.
- **pi2_three_state** — 시간평균이 $\pi$로 가지 않는다: `time_average_occupation`의 시간 가중이 틀렸거나 $\pi$가 $\pi Q=0$의 해가 아니거나($\Pi$의 정상분포를 썼거나) 시작 분포 편향이 남았다.

In [ ]:
assert is_generator(Q3) and is_generator(Q2)
np.testing.assert_allclose(embedded_chain(Q3)[0].sum(1), 1.0)
np.testing.assert_allclose(transition_matrix(Q2, 0.5)[0, 1], exact["p01_at_half"], rtol=1e-10)  # expm vs 닫힌 형식
for k in exact:
    assert_close(estimates[k], exact[k], k=4, name=k)
print("all 5 MC estimates within 4 SE of the closed forms")

## 6. 연습문제

### E1 (계산)

손으로 계산하시오. (a) $Q_3$에서 상태 1의 평균 체류시간과 점프 확률 $\Pi_{10},\Pi_{12}$. (b) 상태 0에서 출발해 '첫 점프가 고장(2)으로 가고 **그리고** 그 점프가 시각 1 이후에 일어날' 확률. (c) 2상태 사슬($\alpha=1,\beta=2$)의 $P_{11}(1)$.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $q_1=1+1=2$이므로 평균 체류시간 $1/q_1=0.5$, $\Pi_{10}=\Pi_{12}=1/2=0.5$.

(b) 정리 1의 **독립성** 덕분에 곱으로 쪼개진다:
$$P(\arg\min=2,\ M>1)=P(\arg\min=2)\,P(M>1)=\frac{q_{02}}{q_0}e^{-q_0\cdot1}=\frac{0.5}{1.5}e^{-1.5}=\frac13e^{-1.5}\approx0.074377.$$
(독립성 없이 "첫 점프가 2로 갈 확률 × 체류가 1을 넘을 확률"이라고 쓸 수 없다 — E3에서 무기억성이 깨지면 이 곱셈도 깨진다.)

(c) 정리 2: $P_{11}(1)=\frac{\alpha}{\alpha+\beta}+\frac{\beta}{\alpha+\beta}e^{-(\alpha+\beta)}=\frac13+\frac23e^{-3}\approx0.366525$.

```python
q0, q1 = -Q3[0, 0], -Q3[1, 1]
print("(a)", 1 / q1, Q3[1, 0] / q1, Q3[1, 2] / q1)
print("(b)", Q3[0, 2] / q0 * np.exp(-q0 * 1.0))
print("(c)", 1/3 + 2/3 * np.exp(-3), transition_matrix(Q2, 1.0)[1, 1])
# (a) 0.5 0.5 0.5 / (b) 0.074377 / (c) 0.366525 0.366525
```

</details>

### E2 (유도 후 시뮬레이션 검증)

2상태 사슬($\alpha=1,\beta=2$)이 0에서 출발할 때 (a) 0으로 처음 되돌아오는 시각($0\to1\to0$)의 기댓값, (b) $[0,2]$ 동안 상태 1에 머문 총 시간의 기댓값 $\int_0^2P_{01}(s)\,ds$를 유도하고, `gillespie_paths(Q2, 0, 2.0, rng, n_paths=8000, n_grid=2001)`의 격자 점유율(사다리꼴 적분) 또는 `time_average_occupation`으로 4 SE 이내 검증하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 0에서의 체류 $\mathrm{Exp}(\alpha)$ + 1에서의 체류 $\mathrm{Exp}(\beta)$의 합(두 체류시간은 독립). 기댓값은 $1/\alpha+1/\beta=1+0.5=1.5$.

(b) 상태 1에 머문 시간은 $\int_0^T\mathbf 1\{X_s=1\}ds$이고 Fubini로 기댓값은 $\int_0^TP_{01}(s)ds$:
$$\int_0^T\frac{\alpha}{\alpha+\beta}\left(1-e^{-(\alpha+\beta)s}\right)ds=\pi_1T-\pi_1\frac{1-e^{-(\alpha+\beta)T}}{\alpha+\beta},\qquad T=2:\ \frac23-\frac{1-e^{-6}}{9}\approx0.555831.$$

```python
n = scale(8_000)
# (a) 첫 귀환 시각: states[1:] 에서 처음 0이 되는 점프의 시각 (T를 넉넉히)
ret = np.empty(n)
for i in range(n):
    times, states = gillespie(Q2, 0, 200.0, rng)
    ret[i] = times[np.flatnonzero(states[1:] == 0)[0] + 1]
r_a = mc_estimate(ret)
assert_close(r_a, 1 / alpha + 1 / beta, k=4, name="E2a first return")
# (b) 두 가지 방법: 격자 점유율의 사다리꼴 적분 / time_average_occupation
T = 2.0
occ_exact = pi1 * T - pi1 * (1 - np.exp(-lam * T)) / lam
t_g, Xg = gillespie_paths(Q2, 0, T, rng, n_paths=n, n_grid=2001)
occ_grid = np.trapezoid((Xg == 1).astype(float), t_g, axis=1)
occ_tao = np.array([time_average_occupation(*gillespie(Q2, 0, T, rng), T, 2)[1] * T for _ in range(n)])
assert_close(mc_estimate(occ_grid), occ_exact, k=4, name="E2b grid")
assert_close(mc_estimate(occ_tao), occ_exact, k=4, name="E2b time_average_occupation")
print(r_a, mc_estimate(occ_grid), mc_estimate(occ_tao), occ_exact)
# 예 (FAST, n=800, 새로 만든 rng_for("03a") 기준): 1.5094 ± 0.0401, 0.5400 ± 0.0161, 0.5635 ± 0.0160, 0.555831
```

</details>

### E3 (가정을 깨보기)

`gillespie_simple`에서 체류시간을 $\mathrm{Exp}(q_i)$ 대신 (i) 상수 $1/q_i$, (ii) $\mathrm{Uniform}(0,2/q_i)$ (평균은 같음)로 바꾼 **준마르코프(semi-Markov)** 과정을 2상태 사슬에 대해 만들고, 0에서 출발한 경로들의 $P_{01}(t)$ 곡선을 정리 2와 비교하시오. 무엇이 깨지는가? 장기 점유율은 어떻게 되는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(i) 상수 체류이면 경로가 **결정론적**이다: 0에 정확히 $1/\alpha=1$, 1에 정확히 $1/\beta=0.5$ 머물므로 $X_t=1\iff t\in[1,1.5)\cup[2.5,3)\cup\dots$ 즉 $P_{01}(0.5)=0$ (정리 2의 $0.258957$ 대신), $P_{01}(1.2)=1$. 지수 곡선과 전혀 다르고 극한도 없다 (주기 1.5로 진동).

(ii) 균등 체류에서는 곡선이 진동하며 수렴하지만 정리 2와 다르다.

**깨진 가정**: 무기억성 $\to$ 마르코프 성질(정리 3) $\to$ 반군 성질 $P(s+t)=P(s)P(t)$ (03b에서 다시 확인). 시각 $t$에 상태를 알아도 '얼마나 오래 머물렀는가'가 미래에 영향을 준다. 반면 **장기 점유율**은 평균 체류시간만으로 정해지므로 두 경우 모두 $(2/3,1/3)$으로 같다 (03c/04c의 재생-보상 논법 예고).

```python
def gillespie_semi(Q, x0, T, rng, hold_sampler):
    """체류시간 분포만 바꾼 준마르코프 버전. hold_sampler(q, rng) -> float"""
    Q = np.asarray(Q, float); n = len(Q)
    t, x = 0.0, int(x0); times, states = [0.0], [x]
    while True:
        q = -Q[x, x]
        if q <= 0:
            break
        t += hold_sampler(q, rng)
        if t > T:
            break
        probs = Q[x].copy(); probs[x] = 0.0; probs /= q
        x = int(rng.choice(n, p=probs))
        times.append(t); states.append(x)
    return np.array(times), np.array(states)

samplers = {"Exp(q)": lambda q, rng: rng.exponential(1 / q),
            "const 1/q": lambda q, rng: 1.0 / q,
            "Unif(0, 2/q)": lambda q, rng: rng.uniform(0, 2 / q)}
T, n = 6.0, scale(2_000)
t_g = np.linspace(0, T, 601)
fig, ax = plt.subplots()
for name, h in samplers.items():
    Xs = np.empty((n, t_g.size), dtype=int)
    for i in range(n):
        times, states = gillespie_semi(Q2, 0, T, rng, h)
        Xs[i] = states[np.searchsorted(times, t_g, side="right") - 1]
    curve = (Xs == 1).mean(0)
    ax.plot(t_g, curve, label=name)
    i05, i12 = np.argmin(abs(t_g - 0.5)), np.argmin(abs(t_g - 1.2))
    print(f"{name:13s} P01(0.5) = {curve[i05]:.3f}  P01(1.2) = {curve[i12]:.3f}  occupation of 1 on [0,6] = {(Xs == 1).mean():.3f}")
ax.plot(t_g, pi1 * (1 - np.exp(-lam * t_g)), "k--", label="Theorem 2")
ax.set_title("P(X_t = 1 | X_0 = 0): exponential vs semi-Markov holding times")
ax.set_xlabel("t"); ax.legend(); plt.show()
# const: P01(0.5) = 0.000, P01(1.2) = 1.000, occupation ~ 0.333 / Unif: P01(0.5) ~ 0.20, P01(1.2) ~ 0.28, occupation ~ 0.31 (T=6은 아직 짧다)
```

</details>

## 7. 정리

**5줄 요약**
1. CTMC는 '상태 $i$에 $\mathrm{Exp}(q_i)$만큼 머문 뒤 $\Pi_{ij}=q_{ij}/q_i$로 점프'하는 과정이며, 경쟁하는 지수 시계 구성과 동치다 (정리 1: 최소는 $\mathrm{Exp}(\sum)$, 승자는 비율에 비례, 둘은 독립).
2. 생성원 $Q$: 비대각 = 비율, 대각 = $-q_i$, 행합 0. $P(h)=I+Qh+o(h)$가 $Q$의 뜻이다.
3. 질레스피 알고리즘은 정확한 표본경로를 만든다 — 오차는 몬테카를로 오차뿐이다 (한 스텝에 난수 2개).
4. 2상태 사슬: $P_{01}(t)=\frac{\alpha}{\alpha+\beta}(1-e^{-(\alpha+\beta)t})$, 수렴 속도는 $\alpha+\beta$, 극한 $\pi=(\beta,\alpha)/(\alpha+\beta)$.
5. 점프 사슬의 정상분포와 CTMC의 정상분포는 다르다 (체류시간 가중); 시간평균 점유율은 $\pi$로 간다 (03c).

**trap 카드**
- Q: 질레스피 시뮬레이션은 시간 격자 $\Delta t$를 잘게 잡아야 정확해지는 근사인가?
- A: 아니다. 체류시간 $\mathrm{Exp}(q_i)$와 점프 $q_{ij}/q_i$를 그대로 샘플링하므로 이산화 오차가 없다. $I+Q\Delta t$로 격자 위에서 굴리는 방식이 근사이며, 그것조차 $q_i\Delta t<1$이어야 확률행렬이 된다. 정확한 격자 전이는 $e^{Q\Delta t}$ (03b).

**다음 노트북: 03b (콜모고로프 방정식과 $P(t)=e^{Qt}$)** — 여기서 얻은 $P(h)=I+Qh+o(h)$와 반군 성질을 미분하면 콜모고로프 전진·후진 방정식이 나오고, 그 해 $P(t)=e^{Qt}$로 임의 시각의 분포를 시뮬레이션 없이 계산할 수 있다 (4.4의 `transition_matrix` 확인이 그 예고였다).

**log/progress.md 한 줄 템플릿**
`- [ ] 03a 생성원·질레스피 — 날짜: ____ / 예측 5개 중 맞은 것: __ / 막힌 곳: ____ / 다음: 03b`